In [ ]:
from pathlib import Path

import pandas as pd

CHEMIN = Path("..") / "data" / "raw" / "regularite_mensuelle_tgv_par_liaisons.csv"
df = pd.read_csv(CHEMIN, sep=";")
df.shape

In [ ]:
df.head()

In [ ]:
df.dtypes

## 1. Profil du jeu de données

In [ ]:
manquants = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
manquants.to_frame("% manquant")

In [ ]:
colonnes_texte = df.select_dtypes(exclude="number").columns
df[colonnes_texte].nunique().sort_values(ascending=False).to_frame("valeurs distinctes")

In [ ]:
print(df["Service"].value_counts())
liaisons = df[["Gare de départ", "Gare d'arrivée"]].drop_duplicates()
print("Nombre de liaisons :", len(liaisons))

In [ ]:
dates = pd.to_datetime(df["Date"], format="%Y-%m")
print("De", dates.min().date(), "à", dates.max().date())
print("Mois distincts :", dates.nunique())
dates.dt.year.value_counts().sort_index()

In [ ]:
for i, col in enumerate(df.columns, start=1):
    exemple = df[col].dropna().iloc[0] if df[col].notna().any() else "vide"
    print(i, "|", col, "|", df[col].dtype, "|", exemple)

## 2. Contrôles qualité

In [ ]:
print("Doublons exacts :", df.duplicated().sum())
cle = ["Date", "Service", "Gare de départ", "Gare d'arrivée"]
print("Doublons fonctionnels (même mois, même liaison) :", df.duplicated(subset=cle).sum())

In [ ]:
print("Circulations prévues = 0 :", (df["Nombre de circulations prévues"] == 0).sum())
for col in df.select_dtypes(exclude="number").columns:
    textes = df[col].dropna().astype(str).str.strip().str.lower()
    n = textes.isin(["", "-1", "non renseigné", "n/a"]).sum()
    if n:
        print(f"{col} : {n} valeurs vides ou déguisées")
for col in df.select_dtypes(include="number").columns:
    n = (df[col] == -1).sum()
    if n:
        print(f"{col} : {n} valeurs égales à -1")

In [ ]:
dates = pd.to_datetime(df["Date"], format="%Y-%m", errors="coerce")
print("Dates illisibles :", dates.isna().sum())

In [ ]:
nombres = df.select_dtypes(include="number")
q1, q3 = nombres.quantile(0.25), nombres.quantile(0.75)
iqr = q3 - q1
hors = ((nombres < q1 - 1.5 * iqr) | (nombres > q3 + 1.5 * iqr)).sum()
display(hors[hors > 0].to_frame("valeurs hors IQR"))

prevues = df["Nombre de circulations prévues"]
print("Annulés > prévus :", (df["Nombre de trains annulés"] > prevues).sum())
print("En retard à l'arrivée > prévus :", (df["Nombre de trains en retard à l'arrivée"] > prevues).sum())
print("Durée moyenne <= 0 :", (df["Durée moyenne du trajet"] <= 0).sum())

In [ ]:
mois = dates.dt.to_period("M")
attendus = pd.period_range(mois.min(), mois.max(), freq="M")
print("Mois absents :", [str(m) for m in attendus.difference(pd.PeriodIndex(mois.unique()))])
lignes_par_mois = df.groupby(mois).size()
print(lignes_par_mois.describe())
print(lignes_par_mois.nsmallest(5))

In [ ]:
sans_circ = df["Nombre de circulations prévues"] == 0
annules_sup = df["Nombre de trains annulés"] > df["Nombre de circulations prévues"]
print("0 circulation ET durée <= 0 :", (sans_circ & (df["Durée moyenne du trajet"] <= 0)).sum())
print("0 circulation ET annulés > 0 :", (sans_circ & (df["Nombre de trains annulés"] > 0)).sum())
print("Annulés > prévus avec prévus > 0 :", (annules_sup & ~sans_circ).sum())
print(df.loc[sans_circ, "Date"].str[:4].value_counts().sort_index())
col30 = "Nombre trains en retard > 30min"
print(df.loc[df[col30] == -1, ["Date", "Service", "Gare de départ", "Gare d'arrivée"]])

In [ ]:
c15, c60 = "Nombre trains en retard > 15min", "Nombre trains en retard > 60min"
valide = df[col30] != -1
print("> 30 supérieur à > 15 :", ((df[col30] > df[c15]) & valide).sum())
print("> 60 supérieur à > 30 :", ((df[c60] > df[col30]) & valide).sum())

causes = [c for c in df.columns if c.startswith("Prct retard pour cause")]
print(len(causes), "colonnes de causes")
somme = df[causes].sum(axis=1)
avec_retard = df["Nombre de trains en retard à l'arrivée"] > 0
print("Somme des causes loin de 100 :", ((somme - 100).abs() > 1)[avec_retard].sum())
print("Pourcentages hors [0 ; 100] :", ((df[causes] < 0) | (df[causes] > 100)).sum().sum())

In [ ]:
liaison = df["Gare de départ"] + " -> " + df["Gare d'arrivée"]
mois_par_liaison = df.groupby(liaison)["Date"].nunique()
print("Liaisons présentes tous les mois :", (mois_par_liaison == df["Date"].nunique()).sum(), "sur", len(mois_par_liaison))

In [ ]:
print("Mois des lignes à 0 circulation :")
print(df.loc[sans_circ, "Date"].value_counts().sort_index())

print("Mois des valeurs -1 :")
print(df.loc[df[col30] == -1, "Date"].value_counts().sort_index())

incoh = (df[c60] > df[col30]) & valide
print("Années des lignes où > 60 dépasse > 30 :")
print(df.loc[incoh, "Date"].str[:4].value_counts().sort_index())

ecart = ((somme - 100).abs() > 1) & avec_retard
print("Somme des causes sur les 89 lignes :")
print(somme[ecart].describe().loc[["min", "50%", "max"]])
print("Dont somme égale à 0 :", (somme[ecart] == 0).sum())

incompletes = mois_par_liaison[mois_par_liaison < df["Date"].nunique()]
print(incompletes.sort_values())

In [ ]:
print("=== E : seuils par année ===")
v = df[df[col30] != -1]
print(v.groupby(v["Date"].str[:4])[[c15, col30, c60]].sum())

print("\n=== F : causes et liaisons incomplètes ===")
print("Causes : dont somme égale à 0 :", (somme[ecart] == 0).sum())
print(incompletes.value_counts().sort_index())

print("\n=== G : gares notées 0 ===")
for col in ["Gare de départ", "Gare d'arrivée"]:
    n = df[col].astype(str).str.strip().isin(["0", ""]).sum()
    print(col, ": valeurs '0' ou vides :", n)
print(df.loc[df["Gare d'arrivée"].astype(str) == "0", "Date"].str[:4].value_counts().sort_index())

print("\n=== H : commentaires ===")
for col in ["Commentaire annulations", "Commentaire retards au départ", "Commentaire retards à l'arrivée"]:
    print(col, ":", df[col].notna().sum(), "lignes remplies sur", len(df))

In [ ]:
print("=== F ===")
print("Causes : dont somme égale à 0 :", (somme[ecart] == 0).sum())
print(incompletes.value_counts().sort_index())

print("\n=== G ===")
for col in ["Gare de départ", "Gare d'arrivée"]:
    n = df[col].astype(str).str.strip().isin(["0", ""]).sum()
    print(col, ": valeurs '0' ou vides :", n)
print(df.loc[df["Gare d'arrivée"].astype(str) == "0", "Date"].str[:4].value_counts().sort_index())